# 반려동물 의상 이미지 수집 파이프라인

이 노트북은 반려동물(고양이/강아지)이 **옷을 입은 이미지**를 Google Images에서 자동 수집하는 도구입니다.

## 전체 흐름

1. 기존 반려동물 이미지 파일명에서 **종(species) 목록**을 추출합니다.
2. 각 종에 대해 **Google Images 검색어**를 만듭니다.
3. Selenium으로 **브라우저를 자동 제어**하여 이미지를 검색합니다.
4. 검색 결과에서 **원본 이미지 URL**을 수집합니다.
5. URL로부터 이미지를 **다운로드하여 저장**합니다.

## 실행 순서

`설정` → `종 추출` → `함수 정의` → `테스트 실행` → `전체 실행` → `결과 확인`

## 1단계: 라이브러리 불러오기

크롤링에 필요한 라이브러리를 불러옵니다.

| 라이브러리 | 용도 |
|---|---|
| `hashlib` | 이미지 중복 검사 (내용 기반 해시) |
| `random` | 대기 시간에 랜덤 변동을 줘서 차단 방지 |
| `time` | 페이지 로딩 대기 |
| `pathlib.Path` | 파일 경로를 안전하게 다루기 |
| `requests` | 이미지 HTTP 다운로드 |
| `selenium` | Chrome 브라우저 자동 제어 |
| `webdriver_manager` | ChromeDriver 자동 설치 |

In [2]:
from __future__ import annotations

import hashlib
import random
import time
from pathlib import Path
from urllib.parse import urlparse

import requests
from selenium import webdriver
from selenium.common.exceptions import (
    ElementClickInterceptedException,
    NoSuchElementException,
    StaleElementReferenceException,
    TimeoutException,
)
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

## 2단계: 경로와 실행 옵션 설정

크롤링 전 **경로**, **수량**, **품질 필터** 등의 설정값을 지정합니다.
이 셀의 값만 바꾸면 함수 내부를 수정하지 않고도 동작을 조정할 수 있습니다.

### 주요 설정

| 변수 | 설명 |
|---|---|
| `SOURCE_DIR` | 기존 반려동물 이미지가 있는 원본 폴더 |
| `OUTPUT_DIR` | 새로 다운로드한 이미지를 저장할 폴더 |
| `LIMIT_PER_SPECIES` | 종당 최대 저장 이미지 수 |
| `MAX_SCROLLS` | 검색 결과 페이지 스크롤 횟수 |

### 품질 필터 설정

| 변수 | 설명 |
|---|---|
| `MIN_FILE_SIZE` | 이 크기보다 작은 이미지는 건너뜀 (썸네일/아이콘 방지) |
| `MAX_DOWNLOAD_RETRIES` | 다운로드 실패 시 재시도 횟수 |

In [3]:
# ── 경로 설정 ──
SOURCE_DIR = Path.cwd().parent / "pet-images" / "images"
OUTPUT_DIR = Path("images")

# ── 수집 설정 ──
LIMIT_PER_SPECIES = 50     # 종당 최대 이미지 수
MAX_SCROLLS = 12           # 결과 페이지 스크롤 횟수
PAUSE = 1.0                # 동작 사이 기본 대기 시간(초)
HEADLESS = False           # True면 브라우저 창 숨김
OVERWRITE = False          # True면 기존 파일 덮어쓰기

# ── 품질 필터 ──
MIN_FILE_SIZE = 5_000      # 최소 파일 크기 (5KB 미만 건너뜀)
MAX_DOWNLOAD_RETRIES = 3   # 다운로드 재시도 횟수
DOWNLOAD_TIMEOUT = 30      # 다운로드 타임아웃(초)

# ── 출력 폴더 생성 ──
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"SOURCE_DIR = {SOURCE_DIR.resolve()}")
print(f"OUTPUT_DIR = {OUTPUT_DIR.resolve()}")

SOURCE_DIR = C:\Users\master\02_Workspace\Google_Pet To\pet-images\images
OUTPUT_DIR = C:\Users\master\02_Workspace\Google_Pet To\clothes-images\images


## 3단계: 종(species) 목록 추출

`pet-images/images/cat`과 `pet-images/images/dog` 폴더에 있는
이미지 파일명에서 종 이름을 추출합니다.

**예시:**

| 파일 경로 | 추출 결과 |
|---|---|
| `cat/Abyssinian_12.jpg` | `species="Abyssinian"`, `animal="cat"` |
| `dog/beagle_34.jpg` | `species="beagle"`, `animal="dog"` |

In [6]:
def extract_species_list(source_dir: Path) -> list[dict[str, str]]:
    """원본 이미지 폴더에서 종(species) 목록을 추출합니다."""
    seen = set()
    species_list = []

    for animal_type in ["cat", "dog"]:
        animal_dir = source_dir / animal_type

        # 폴더가 없으면 건너뛰기
        if not animal_dir.exists():
            print(f"  폴더 없음, 건너뜀: {animal_dir}")
            continue

        for image_file in animal_dir.glob("*.jpg"):
            # 파일명에서 마지막 번호를 제거하여 종 이름 추출
            # 예: "Abyssinian_12" → ["Abyssinian", "12"] → "Abyssinian"
            name_parts = image_file.stem.split("_")
            species_name = "_".join(name_parts[:-1])

            if not species_name:
                continue

            # 같은 종은 한 번만 기록
            key = (species_name, animal_type)
            if key in seen:
                continue

            seen.add(key)
            species_list.append({
                "species": species_name,
                "animal": animal_type,
            })

    # 동물 타입 → 종 이름 순으로 정렬
    species_list.sort(key=lambda x: (x["animal"], x["species"]))
    return species_list


# 종 목록 추출 실행
species_list = extract_species_list(SOURCE_DIR)
print(f"총 {len(species_list)}개 종 발견\n")
print(", ".join(f"{record['species']} ({record['animal']})" for record in species_list))

총 37개 종 발견

Abyssinian (cat), Bengal (cat), Birman (cat), Bombay (cat), British_Shorthair (cat), Egyptian_Mau (cat), Maine_Coon (cat), Persian (cat), Ragdoll (cat), Russian_Blue (cat), Siamese (cat), Sphynx (cat), american_bulldog (dog), american_pit_bull_terrier (dog), basset_hound (dog), beagle (dog), boxer (dog), chihuahua (dog), english_cocker_spaniel (dog), english_setter (dog), german_shorthaired (dog), great_pyrenees (dog), havanese (dog), japanese_chin (dog), keeshond (dog), leonberger (dog), miniature_pinscher (dog), newfoundland (dog), pomeranian (dog), pug (dog), saint_bernard (dog), samoyed (dog), scottish_terrier (dog), shiba_inu (dog), staffordshire_bull_terrier (dog), wheaten_terrier (dog), yorkshire_terrier (dog)


## 4단계: 검색어 생성 및 유틸리티 함수

### 검색어 개선

기존 검색어 `"Abyssinian cat clothes"`는 **쇼핑 상품 이미지**가 많이 섞이는 문제가 있었습니다.

개선된 검색어 `"Abyssinian cat wearing clothes"`를 사용하면
**실제로 옷을 입은 동물 이미지**가 더 많이 검색됩니다.

### 랜덤 대기

매번 같은 시간을 기다리면 봇으로 탐지될 수 있습니다.
`wait_randomly()`로 대기 시간에 변동을 줘서 자연스러운 브라우징을 시뮬레이션합니다.

In [4]:
def build_search_query(species_name: str, animal_type: str) -> str:
    """종 이름과 동물 타입으로 Google Images 검색어를 만듭니다.

    'wearing clothes'를 사용하면 옷을 입은 동물 이미지가 더 잘 나옵니다.
    단순히 'clothes'만 쓰면 쇼핑 상품 이미지가 많이 섞입니다.
    """
    readable_name = species_name.replace("_", " ")
    return f"{readable_name} {animal_type} wearing clothes"


def wait_randomly(min_seconds: float, max_seconds: float) -> None:
    """min_seconds ~ max_seconds 사이 랜덤 시간만큼 대기합니다."""
    time.sleep(random.uniform(min_seconds, max_seconds))


# 검색어 예시 출력
sample = species_list[0]
print(f"종: {sample['species']}, 동물: {sample['animal']}")
print(f"검색어: {build_search_query(sample['species'], sample['animal'])}")

종: Abyssinian, 동물: cat
검색어: Abyssinian cat wearing clothes


## 5단계: 브라우저 제어 함수

Selenium으로 Chrome 브라우저를 자동 제어하는 함수들입니다.

| 함수 | 역할 |
|---|---|
| `create_browser()` | Chrome 브라우저 생성 |
| `handle_consent_popup()` | Google 쿠키 동의 팝업 처리 |
| `open_google_images()` | Google Images 페이지 열기 |
| `search_images()` | 검색어 입력 및 검색 실행 |

In [5]:
def create_browser(headless: bool = False):
    """Selenium Chrome 브라우저를 생성합니다."""
    options = Options()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--log-level=3")
    options.add_argument("--start-maximized")
    options.add_argument("--window-size=1600,1200")
    if headless:
        options.add_argument("--headless=new")

    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)


def handle_consent_popup(driver, pause: float = 1.0) -> None:
    """Google 쿠키 동의 팝업이 뜨면 자동으로 닫아줍니다."""
    consent_xpaths = [
        "//button[.//div[text()='I agree']]",
        "//button[.//div[text()='Accept all']]",
        "//button[.='I agree']",
        "//button[.='Accept all']",
        "//button[contains(@aria-label, 'Accept')]",
        "//button[contains(@id, 'L2AGLb')]",
    ]
    for xpath in consent_xpaths:
        try:
            button = WebDriverWait(driver, 2).until(
                EC.element_to_be_clickable((By.XPATH, xpath))
            )
            button.click()
            time.sleep(pause)
            return
        except (TimeoutException, ElementClickInterceptedException):
            continue


def open_google_images(driver, pause: float = 1.0) -> None:
    """Google Images 메인 페이지를 열고 동의 팝업을 처리합니다."""
    driver.get("https://images.google.com/")
    time.sleep(pause)
    handle_consent_popup(driver, pause)


def search_images(driver, query: str, pause: float = 1.0) -> None:
    """검색창에 검색어를 입력하고 검색을 실행합니다."""
    search_box = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.NAME, "q"))
    )
    search_box.clear()
    search_box.send_keys(query)
    search_box.send_keys(Keys.ENTER)
    wait_randomly(pause, pause * 2)

## 6단계: 이미지 URL 수집

검색 결과 페이지에서 **원본 이미지 URL**을 수집하는 함수들입니다.

### 수집 흐름

1. 검색 결과의 **썸네일 목록**을 가져옵니다.
2. 아직 클릭하지 않은 썸네일을 **하나씩 클릭**합니다.
3. 클릭하면 나타나는 **원본 이미지 URL**을 추출합니다.
4. 유효한 URL만 모아서 반환합니다.
5. 더 많은 결과가 필요하면 **스크롤**합니다.

### 품질 필터

- `data:` (base64 인코딩) URL 제외
- Google 자체 이미지 (gstatic.com 등) 제외
- 중복 URL 제외

In [6]:
def is_valid_image_url(url: str) -> bool:
    """다운로드 가능한 실제 이미지 URL인지 확인합니다."""
    if not url or url.startswith("data:"):
        return False
    if not url.startswith(("http://", "https://")):
        return False

    # Google 자체 이미지(로고, 아이콘 등) 제외
    skip_patterns = ["gstatic.com/images", "google.com/images", "googleapis.com"]
    for pattern in skip_patterns:
        if pattern in url:
            return False

    return True


def click_show_more(driver, pause: float = 1.0) -> bool:
    """'더보기' 버튼이 있으면 클릭합니다. 성공 시 True를 반환합니다."""
    xpaths = [
        "//input[@type='button' and @value='Show more results']",
        "//button[contains(., 'Show more results')]",
        "//a[contains(., 'Show more results')]",
    ]
    for xpath in xpaths:
        try:
            button = driver.find_element(By.XPATH, xpath)
            driver.execute_script("arguments[0].click();", button)
            time.sleep(pause)
            return True
        except NoSuchElementException:
            continue
    return False


def extract_full_image_url(driver) -> str | None:
    """클릭된 썸네일의 원본 이미지 URL을 추출합니다.

    Google Images는 클래스명이 자주 바뀌므로 여러 선택자를 순서대로 시도합니다.
    동작하지 않으면 아래 리스트에 새 선택자를 추가해 주세요.
    """
    selectors = [
        "img.sFlh5c.FyHeAf",
        "img.n3VNCb",
        "img.iPVvYb",
        "img.sFlh5c",
    ]
    for selector in selectors:
        try:
            candidates = driver.find_elements(By.CSS_SELECTOR, selector)
            for img in candidates:
                src = img.get_attribute("src") or ""
                if is_valid_image_url(src):
                    return src
        except StaleElementReferenceException:
            continue
    return None


def collect_image_urls(
    driver,
    limit: int,
    max_scrolls: int,
    pause: float = 1.0,
) -> list[str]:
    """검색 결과 페이지에서 원본 이미지 URL을 수집합니다.

    매 스크롤 라운드마다 썸네일 목록을 새로 가져오고,
    이미 클릭한 썸네일은 건너뛰어 중복 작업을 방지합니다.
    """
    collected_urls: list[str] = []
    seen_urls: set[str] = set()
    clicked_thumbs: set[str] = set()

    for scroll_round in range(max_scrolls):
        if len(collected_urls) >= limit:
            break

        # 현재 화면의 썸네일 목록 (매 라운드 새로 가져옴)
        thumbnails = driver.find_elements(
            By.CSS_SELECTOR, "img.rg_i, img.YQ4gaf"
        )

        for thumbnail in thumbnails:
            if len(collected_urls) >= limit:
                break

            # 이미 클릭한 썸네일인지 확인
            try:
                thumb_id = (
                    thumbnail.get_attribute("src")
                    or thumbnail.get_attribute("data-src")
                    or ""
                )
            except StaleElementReferenceException:
                continue

            if thumb_id in clicked_thumbs:
                continue
            clicked_thumbs.add(thumb_id)

            # 썸네일 클릭하여 원본 이미지 패널 열기
            try:
                driver.execute_script(
                    "arguments[0].scrollIntoView({block: 'center'});",
                    thumbnail,
                )
                wait_randomly(pause * 0.3, pause * 0.6)
                driver.execute_script("arguments[0].click();", thumbnail)
                wait_randomly(pause * 0.8, pause * 1.5)
            except (
                ElementClickInterceptedException,
                StaleElementReferenceException,
            ):
                continue

            # 원본 이미지 URL 추출
            full_url = extract_full_image_url(driver)
            if full_url and full_url not in seen_urls:
                seen_urls.add(full_url)
                collected_urls.append(full_url)

        # 아래로 스크롤하여 더 많은 결과 로드
        driver.execute_script(
            "window.scrollBy(0, document.body.scrollHeight);"
        )
        wait_randomly(pause, pause * 1.5)
        click_show_more(driver, pause)

    return collected_urls

## 7단계: 이미지 다운로드 및 저장

수집한 URL에서 실제 이미지를 다운로드하고 종별 폴더에 저장합니다.

### 품질 보장 기능

| 기능 | 설명 |
|---|---|
| **재시도** | 네트워크 오류 시 최대 3회 재시도 |
| **크기 필터** | 5KB 미만 파일 건너뜀 (썸네일/아이콘 방지) |
| **중복 방지** | 이미지 내용(MD5 해시) 기반 중복 검사 |
| **이어쓰기** | 기존 파일 번호 이어서 저장 (덮어쓰기 방지) |
| **확장자 추론** | 응답 헤더의 `content-type`으로 정확한 확장자 결정 |

In [7]:
def guess_extension(response, url: str) -> str:
    """HTTP 응답 또는 URL에서 이미지 확장자를 추론합니다."""
    content_type = (
        response.headers.get("content-type", "").split(";")[0].strip().lower()
    )
    type_to_ext = {
        "image/jpeg": ".jpg",
        "image/png": ".png",
        "image/webp": ".webp",
        "image/gif": ".gif",
    }
    if content_type in type_to_ext:
        return type_to_ext[content_type]

    # content-type으로 판별 못하면 URL 경로에서 추론
    url_suffix = Path(urlparse(url).path).suffix.lower()
    if url_suffix in {".jpg", ".jpeg", ".png", ".webp", ".gif"}:
        return url_suffix

    return ".jpg"


def download_image(url: str) -> dict | None:
    """URL에서 이미지를 다운로드합니다. 실패 시 None을 반환합니다.

    - content-type이 image/가 아니면 건너뜀
    - MIN_FILE_SIZE보다 작으면 건너뜀 (썸네일/아이콘 방지)
    - 실패 시 MAX_DOWNLOAD_RETRIES만큼 재시도
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    for attempt in range(MAX_DOWNLOAD_RETRIES):
        try:
            response = requests.get(
                url, timeout=DOWNLOAD_TIMEOUT, headers=headers
            )
            response.raise_for_status()

            # 이미지 응답인지 확인
            content_type = response.headers.get("content-type", "").lower()
            if not content_type.startswith("image/"):
                return None

            image_data = response.content

            # 너무 작은 파일은 건너뛰기 (썸네일, 아이콘 등)
            if len(image_data) < MIN_FILE_SIZE:
                return None

            extension = guess_extension(response, url)
            return {"data": image_data, "extension": extension}

        except requests.RequestException:
            if attempt < MAX_DOWNLOAD_RETRIES - 1:
                wait_randomly(1, 3)

    return None


def save_images_for_species(
    species_name: str,
    animal_type: str,
    image_urls: list[str],
    output_dir: Path,
    overwrite: bool = False,
) -> int:
    """수집한 URL들에서 이미지를 다운로드하여 종별 폴더에 저장합니다.

    - 기존 파일의 내용 해시로 중복 다운로드를 방지합니다.
    - 파일 번호는 기존 파일 이후부터 이어서 매깁니다.
    """
    save_dir = output_dir / species_name
    save_dir.mkdir(parents=True, exist_ok=True)

    # 기존 파일의 내용 해시 수집 (중복 다운로드 방지)
    existing_hashes: set[str] = set()
    for existing_file in save_dir.glob("*"):
        if existing_file.is_file():
            file_hash = hashlib.md5(existing_file.read_bytes()).hexdigest()
            existing_hashes.add(file_hash)

    # 기존 파일의 최대 번호 찾기 → 이어서 번호 매기기
    max_number = 0
    for existing_file in save_dir.glob(f"{species_name}_{animal_type}_*"):
        try:
            number = int(existing_file.stem.split("_")[-1])
            max_number = max(max_number, number)
        except ValueError:
            pass

    next_number = max_number + 1
    saved_count = 0

    for url in image_urls:
        result = download_image(url)
        if result is None:
            continue

        # 내용 기반 중복 확인
        content_hash = hashlib.md5(result["data"]).hexdigest()
        if content_hash in existing_hashes:
            continue
        existing_hashes.add(content_hash)

        # 파일 저장
        filename = (
            f"{species_name}_{animal_type}_{next_number:03d}"
            f"{result['extension']}"
        )
        file_path = save_dir / filename

        if file_path.exists() and not overwrite:
            next_number += 1
            continue

        file_path.write_bytes(result["data"])
        next_number += 1
        saved_count += 1

    return saved_count

## 8단계: 한 종만 테스트 실행

전체를 돌리기 전에 **한 종만 먼저 테스트**하여 정상 동작을 확인합니다.

- 브라우저가 정상적으로 열리는지
- Google Images 검색이 되는지
- 이미지 URL이 수집되는지
- 다운로드 및 저장이 정상인지

> **`saved_count`가 0이면** 네트워크 상태, CSS 선택자, 또는 Google DOM 구조 변경을 확인해야 합니다.

In [8]:
# 테스트할 종 선택
test_record = species_list[0]
test_species = test_record["species"]
test_animal = test_record["animal"]
test_query = build_search_query(test_species, test_animal)

print(f"테스트 종: {test_species} ({test_animal})")
print(f"검색어: {test_query}")
print()

# 브라우저 열기 → 검색 → URL 수집 → 다운로드
driver = create_browser(headless=HEADLESS)
open_google_images(driver, PAUSE)
search_images(driver, test_query, PAUSE)

test_urls = collect_image_urls(driver, limit=5, max_scrolls=3, pause=PAUSE)

saved_count = save_images_for_species(
    test_species, test_animal, test_urls, OUTPUT_DIR, overwrite=OVERWRITE
)

print(f"수집 URL: {len(test_urls)}개")
print(f"저장 이미지: {saved_count}개")

테스트 종: Abyssinian (cat)
검색어: Abyssinian cat wearing clothes

수집 URL: 0개
저장 이미지: 0개


## 9단계: 테스트 브라우저 닫기

테스트에 사용한 브라우저 세션을 정리합니다.
`driver.quit()`는 브라우저 창뿐 아니라 Selenium 세션 전체를 종료합니다.

In [9]:
driver.quit()

## 10단계: 전체 종 일괄 수집

모든 종에 대해 `검색 → URL 수집 → 다운로드`를 반복 실행합니다.

- `try ... finally`로 오류가 나더라도 브라우저가 항상 정리됩니다.
- 각 종마다 진행 상태(`[현재/전체]`)를 출력합니다.
- 특정 종만 처리하려면 `TARGET_RECORDS`를 필터링하세요.

```python
# 고양이만 처리하는 예시
TARGET_RECORDS = [r for r in species_list if r["animal"] == "cat"]
```

In [10]:
# 대상 종 목록 (전체 또는 필터링)
TARGET_RECORDS = species_list

total = len(TARGET_RECORDS)
driver = create_browser(headless=HEADLESS)

try:
    for idx, record in enumerate(TARGET_RECORDS, start=1):
        species_name = record["species"]
        animal_type = record["animal"]
        query = build_search_query(species_name, animal_type)

        print(f"\n[{idx}/{total}] {species_name} ({animal_type})")
        print(f"  검색어: {query}")

        open_google_images(driver, PAUSE)
        search_images(driver, query, PAUSE)

        image_urls = collect_image_urls(
            driver, LIMIT_PER_SPECIES, MAX_SCROLLS, PAUSE
        )
        saved_count = save_images_for_species(
            species_name, animal_type, image_urls, OUTPUT_DIR,
            overwrite=OVERWRITE,
        )

        print(f"  수집 URL: {len(image_urls)}개 → 저장: {saved_count}개")

finally:
    driver.quit()
    print("\n브라우저가 정리되었습니다.")


[1/37] Abyssinian (cat)
  검색어: Abyssinian cat wearing clothes
  수집 URL: 50개 → 저장: 50개

[2/37] Bengal (cat)
  검색어: Bengal cat wearing clothes
  수집 URL: 38개 → 저장: 38개

[3/37] Birman (cat)
  검색어: Birman cat wearing clothes
  수집 URL: 29개 → 저장: 29개

[4/37] Bombay (cat)
  검색어: Bombay cat wearing clothes
  수집 URL: 50개 → 저장: 48개

[5/37] British_Shorthair (cat)
  검색어: British Shorthair cat wearing clothes
  수집 URL: 46개 → 저장: 45개

[6/37] Egyptian_Mau (cat)
  검색어: Egyptian Mau cat wearing clothes
  수집 URL: 50개 → 저장: 50개

[7/37] Maine_Coon (cat)
  검색어: Maine Coon cat wearing clothes
  수집 URL: 41개 → 저장: 41개

[8/37] Persian (cat)
  검색어: Persian cat wearing clothes

브라우저가 정리되었습니다.


KeyboardInterrupt: 

## 11단계: 저장 결과 확인

다운로드가 완료된 후 종별 저장 결과를 요약합니다.
특정 종의 파일 수가 0이면 해당 종의 검색 결과가 부족했거나 다운로드 실패가 많았음을 의미합니다.

In [11]:
print(f"{'종 이름':<35} {'파일 수':>6}")
print("-" * 43)

total_images = 0
for species_dir in sorted(OUTPUT_DIR.iterdir()):
    if species_dir.is_dir():
        count = len(list(species_dir.glob("*")))
        total_images += count
        print(f"{species_dir.name:<35} {count:>6}")

print("-" * 43)
print(f"{'합계':<35} {total_images:>6}")

종 이름                                  파일 수
-------------------------------------------
Abyssinian                              50
Bengal                                  38
Birman                                  29
Bombay                                  48
British_Shorthair                       45
cat                                      0
dog                                      0
Egyptian_Mau                            50
Maine_Coon                              41
-------------------------------------------
합계                                     301
